# Stock Price Prediction (Next-Day)

Predicts AAPL's next-day RETURN from lag/technical features (real Yahoo Finance data, 10 years), converts it to an implied price, and benchmarks honestly against the naive baseline.


In [1]:

import io, json, urllib.request
import pandas as pd, numpy as np
# Real AAPL daily prices (10 years) from Yahoo Finance's chart API - no key needed.
req = urllib.request.Request("https://query1.finance.yahoo.com/v8/finance/chart/AAPL?range=10y&interval=1d", headers={"User-Agent": "Mozilla/5.0"})
j = json.loads(urllib.request.urlopen(req).read())
res0 = j["chart"]["result"][0]; q = res0["indicators"]["quote"][0]
s = pd.DataFrame({"Date": pd.to_datetime(res0["timestamp"], unit="s"), "Close": q["close"]}).dropna().sort_values("Date").reset_index(drop=True)
print("rows:", len(s), "| from", s["Date"].min().date(), "to", s["Date"].max().date())


rows: 2514 | from 2016-10-24 to 2026-10-23


In [2]:

# Features are returns/ratios (stationary), and the target is TOMORROW'S RETURN.
# Predicting the price level directly would just memorise the price range of the
# training years and fail later - predicting returns is the honest design.
s["ret1"] = s["Close"].pct_change(1); s["ret5"] = s["Close"].pct_change(5); s["ret10"] = s["Close"].pct_change(10)
s["ma10"] = s["Close"].rolling(10).mean() / s["Close"] - 1
s["ma30"] = s["Close"].rolling(30).mean() / s["Close"] - 1
s["vol10"] = s["ret1"].rolling(10).std()
s["target_ret"] = s["Close"].pct_change().shift(-1)
s2 = s.dropna().reset_index(drop=True)
feats = ["ret1", "ret5", "ret10", "ma10", "ma30", "vol10"]
cut = int(len(s2) * 0.8)   # chronological split - never shuffle a time series
X_train, X_test = s2[feats][:cut], s2[feats][cut:]
y_train = s2["target_ret"][:cut]
today_close = s2["Close"][cut:].values
actual_next = s2["Close"].shift(-1)[cut:].values
actual_ret = s2["target_ret"][cut:].values
today_close, actual_next, actual_ret = today_close[:-1], actual_next[:-1], actual_ret[:-1]  # last row has no next day
X_test = X_test[:-1]
print("train days:", len(X_train), "| test days:", len(X_test))


train days: 1987 | test days: 496


In [3]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
naive_mae = float(np.mean(np.abs(actual_next - today_close)))
print("Naive baseline (tomorrow = today) MAE: $", round(naive_mae, 2))
for name, m in {"Linear Regression": LinearRegression(),
                "Random Forest": RandomForestRegressor(n_estimators=250, random_state=42, n_jobs=-1)}.items():
    m.fit(X_train, y_train); pr = m.predict(X_test)
    implied = today_close * (1 + pr)
    mae = float(mean_absolute_error(actual_next, implied))
    dir_acc = float(np.mean(np.sign(pr) == np.sign(actual_ret)))
    print(name, "implied-price MAE: $", round(mae, 2), "| direction accuracy:", round(dir_acc, 4))


Naive baseline (tomorrow = today) MAE: $ 3.04
Linear Regression implied-price MAE: $ 3.07 | direction accuracy: 0.4879


Random Forest implied-price MAE: $ 3.27 | direction accuracy: 0.4899


## Honest note

Next-day stock moves are close to a random walk: expect the models to roughly MATCH the naive baseline on MAE and sit near 50% on direction. That is the realistic, publishable result — this project teaches correct time-series practice (chronological splits, leakage-free features, return targets), it is NOT investment advice and cannot predict the market.
